In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb


In [6]:
df_2024 = pd.read_csv("../data/df_2024_final.csv").assign(season=2024)
df_2025 = pd.read_csv("../data/df_2025_final.csv").assign(season=2025)
df = pd.concat([df_2024, df_2025], ignore_index=True)

In [13]:
df.head()
df= df.drop(columns="Unnamed: 0")

In [14]:


id_cols = ['player_id', 'player', 'position', 'team', 'season', 'week',
           'date', 'opponent', 'atVs']

context_cols = ['opp_def_rank', 'team_pass_rank', 'team_rush_rank',
                'stadium', 'roof', 'surface', 'temp', 'wind', 'location',
                'gametime']

target_cols = ['fantasy_points', 'projected_points', 'med_proj_pts']

# Post-game outcome columns: keep for EDA, exclude from predictive models
outcome_cols = ['result', 'score']

rushing = ['rushingAttempts', 'rushingYards', 'yardsPerRushAttempt',
           'rushingTouchdowns', 'longRushing']
receiving = ['receptions', 'receivingTargets', 'receivingYards',
             'yardsPerReception', 'receivingTouchdowns', 'longReception']
ball_security = ['fumbles', 'fumblesLost']
passing = ['completions', 'passingAttempts', 'passingYards', 'completionPct',
           'yardsPerPassAttempt', 'passingTouchdowns', 'interceptions',
           'longPassing', 'sacks', 'QBRating', 'adjQBR']
kicking = ['fieldGoalsMade', 'fieldGoalAttempts', 'fieldGoalPct',
           'longFieldGoalMade', 'fieldGoalsMadeYardsAverage',
           'fieldGoalsMade1_19', 'fieldGoalAttempts1_19',
           'fieldGoalsMade20_29', 'fieldGoalAttempts20_29',
           'fieldGoalsMade30_39', 'fieldGoalAttempts30_39',
           'fieldGoalsMade40_49', 'fieldGoalAttempts40_49',
           'fieldGoalsMade50', 'fieldGoalAttempts50',
           'extraPointsMade', 'extraPointAttempts', 'totalKickingPoints']

base = id_cols + context_cols + target_cols + outcome_cols

position_cols = {
    'QB': base + passing + rushing + ball_security,
    'WR': base + receiving + rushing + ball_security,
    'TE': base + receiving + rushing + ball_security,
    'K':  base + kicking,
    "RB": base + rushing + receiving + ball_security
}

pos_dfs = {pos: df.loc[df['position'] == pos, cols].copy()
           for pos, cols in position_cols.items()}

qb_df, wr_df, te_df, k_df, rb_df = (pos_dfs[p] for p in ['QB', 'WR', 'TE', 'K', 'RB'])

In [ ]:
## combined data frames based on cluster
qb_te = pd.concat([qb_df, te_df])
#qb_te.head()


,player_id,player,position,team,season,week,date,opponent,atVs,opp_def_rank,...,rushingTouchdowns,longRushing,fumbles,fumblesLost,receptions,receivingTargets,receivingYards,yardsPerReception,receivingTouchdowns,longReception
85,3916387,Lamar Jackson,QB,BAL,2024,18,2025-01-04 21:30:00+00:00,CLE,home,27,...,0.0,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86,3916387,Lamar Jackson,QB,BAL,2024,17,2024-12-25 21:30:00+00:00,HOU,away,14,...,1.0,48.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
87,3916387,Lamar Jackson,QB,BAL,2024,16,2024-12-21 21:30:00+00:00,PIT,home,8,...,0.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
88,3916387,Lamar Jackson,QB,BAL,2024,15,2024-12-15 18:00:00+00:00,NYG,away,21,...,0.0,21.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
89,3916387,Lamar Jackson,QB,BAL,2024,13,2024-12-01 21:25:00+00:00,PHI,home,2,...,0.0,39.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
wr_k = pd.concat([wr_df, k_df])